In [0]:
-- ---------- AGGREGATES ----------


In [0]:
-- Team consistency metrics (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_consistency_metrics AS
SELECT league_id, season, roster_id,
       stddev_pop(points_for) AS points_for_stddev,
       avg(points_for)       AS points_for_avg,
       count(*)              AS games_played,
       CASE WHEN avg(points_for) <> 0 THEN stddev_pop(points_for)/avg(points_for) END AS points_for_cv
FROM fact_team_week
GROUP BY league_id, season, roster_id;


In [0]:
-- Head to head records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_rivalry_head_to_head AS
WITH a AS (SELECT * FROM fact_team_week),
     b AS (SELECT * FROM fact_team_week)
SELECT a.league_id, a.season,
       a.roster_id AS roster_id_a, b.roster_id AS roster_id_b,
       count(*) AS games_played,
       sum(CASE WHEN a.points_for > b.points_for THEN 1 ELSE 0 END) AS wins_a,
       sum(CASE WHEN a.points_for < b.points_for THEN 1 ELSE 0 END) AS wins_b,
       sum(CASE WHEN a.points_for = b.points_for THEN 1 ELSE 0 END) AS ties,
       avg(a.points_for) AS pf_per_game_a,
       avg(b.points_for) AS pf_per_game_b
FROM a JOIN b
  ON a.league_id=b.league_id AND a.season=b.season AND a.week=b.week
 AND a.roster_id<>b.roster_id
GROUP BY a.league_id, a.season, a.roster_id, b.roster_id;


In [0]:
-- All-time records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_records_all_time AS
SELECT league_id, roster_id,
       max(points_for) AS max_points_for,
       min(points_for) AS min_points_for
FROM fact_team_week
GROUP BY league_id, roster_id;


In [0]:
-- Player total points per roster (answers "most points scored for team")
CREATE OR REPLACE MATERIALIZED VIEW agg_player_roster_totals AS
SELECT 
  pw.league_id,
  pw.roster_id,
  pw.player_id,
  p.full_name,
  p.position,
  min(pw.season) AS first_season,
  max(pw.season) AS last_season,
  count(DISTINCT pw.season) AS seasons_count,
  count(*) AS weeks_count,
  sum(pw.points) AS total_points,
  avg(pw.points) AS points_per_week,
  sum(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS points_as_starter
FROM fact_player_week pw
LEFT JOIN dim_players p ON pw.player_id = p.player_id
GROUP BY pw.league_id, pw.roster_id, pw.player_id, p.full_name, p.position;

In [0]:
-- Draft ROI by round (enhanced with career tracking)
CREATE OR REPLACE MATERIALIZED VIEW agg_draft_roi_by_round AS
WITH picks AS (
  SELECT
    p.league_id,
    li.season,
    CAST(p.round AS INT) AS round,
    p.player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot p
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
player_season_points AS (
  SELECT
    league_id, season, player_id,
    SUM(points) AS season_points
  FROM fact_player_week
  GROUP BY league_id, season, player_id
),
player_career_points AS (
  SELECT
    league_id, player_id,
    SUM(points) AS career_points
  FROM fact_player_week
  GROUP BY league_id, player_id
),
rook AS (
  SELECT
    pk.league_id, pk.season, pk.round, pk.player_id,
    COALESCE(psp.season_points, 0.0) AS rookie_season_points,
    COALESCE(pcp.career_points, 0.0) AS career_points
  FROM picks pk
  LEFT JOIN player_season_points psp
    ON pk.league_id = psp.league_id
   AND pk.season    = psp.season
   AND pk.player_id = psp.player_id
  LEFT JOIN player_career_points pcp
    ON pk.league_id = pcp.league_id
   AND pk.player_id = pcp.player_id
),
by_round AS (
  SELECT
    league_id, season, round,
    AVG(rookie_season_points) AS avg_rookie_points,
    AVG(career_points) AS avg_career_points,
    COUNT(*) AS picks_count
  FROM rook
  GROUP BY league_id, season, round
),
season_avg AS (
  SELECT
    league_id, season,
    AVG(rookie_season_points) AS season_avg_points,
    AVG(career_points) AS career_avg_points
  FROM rook
  GROUP BY league_id, season
)
SELECT
  b.league_id,
  b.season,
  b.round,
  b.picks_count,
  b.avg_rookie_points,
  b.avg_career_points,
  s.season_avg_points,
  s.career_avg_points,
  CASE WHEN s.season_avg_points > 0
       THEN b.avg_rookie_points / s.season_avg_points
       ELSE NULL
  END AS rookie_roi,
  CASE WHEN s.career_avg_points > 0
       THEN b.avg_career_points / s.career_avg_points
       ELSE NULL
  END AS career_roi
FROM by_round b
JOIN season_avg s
  ON b.league_id = s.league_id AND b.season = s.season;
